# Capstone — Feature Engineering & Label Definition

Builds on EDA findings. Creates engineered features, defines multiple label formulations,
and checks for leakage.

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import json
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)

ROOT = Path.cwd().parent.parent
RAW_PATH = ROOT / 'data' / 'raw' / 'content_refresh_anonymized.csv'
PROCESSED_DIR = ROOT / 'data' / 'processed'
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

df = pd.read_csv(RAW_PATH)
print(f'Loaded: {df.shape}')

## 1. Label Definitions

Three formulations:
1. **Binary** `is_declining` — `trend_direction == 'down'` (existing)
2. **3-class** `trend_class` — growing / stable / declining
3. **Regression** `trend_pct` — continuous decline magnitude

In [ ]:
# Binary: is_declining
df['is_declining'] = (df['trend_direction'] == 'down').astype(int)

# 3-class: growing (up/new), stable (stable/flat), declining (down)
trend_map = {'up': 'growing', 'new': 'growing', 'stable': 'stable', 'flat': 'stable', 'down': 'declining'}
df['trend_class'] = df['trend_direction'].map(trend_map)

# Regression: trend_pct (already in data, fill NaN with 0)
df['trend_pct_filled'] = pd.to_numeric(df['trend_pct'], errors='coerce').fillna(0)

# Alternative binary thresholds
for thresh in [10, 20, 30]:
    df[f'decline_{thresh}pct'] = (df['trend_pct_filled'] <= -thresh).astype(int)

print('Label distributions:')
print(f'  is_declining: {df["is_declining"].mean():.3f} (positive rate)')
print(f'  trend_class: {df["trend_class"].value_counts().to_dict()}')
print(f'  trend_pct_filled: mean={df["trend_pct_filled"].mean():.1f}, std={df["trend_pct_filled"].std():.1f}')
for thresh in [10, 20, 30]:
    print(f'  decline_{thresh}pct: {df[f"decline_{thresh}pct"].mean():.3f}')

## 2. Feature Engineering

### Leakage guard
**Never use as features:** `trend_direction`, `trend_pct`, `is_declining`, `trend_class`,
and any `decline_*pct` columns — these are labels.

In [ ]:
LEAKAGE_COLS = [
    'trend_direction', 'trend_pct', 'trend_pct_filled',
    'is_declining', 'trend_class',
    'decline_10pct', 'decline_20pct', 'decline_30pct',
]

# --- Numeric conversions ---
numeric_raw = [
    'search_volume', 'competition', 'cpc',
    'word_count', 'char_count',
    'impressions_90d', 'clicks_90d', 'pageviews_90d',
    'sessions_90d', 'users_90d', 'engaged_sessions_90d',
    'ai_sessions_90d', 'scroll_events_90d',
    'days_with_impressions', 'days_with_sessions',
    'impressions_last_30d', 'clicks_last_30d', 'sessions_last_30d',
    'impressions_prev_30d', 'clicks_prev_30d', 'sessions_prev_30d',
    'content_age_days', 'days_since_last_update',
    'ctr', 'avg_position', 'engagement_rate', 'scroll_rate', 'ai_traffic_pct',
]
for col in numeric_raw:
    if col in df.columns:
        df[col] = pd.to_numeric(df[col], errors='coerce')

# --- Log transforms (heavy-tailed traffic) ---
log_cols = ['impressions_90d', 'clicks_90d', 'sessions_90d', 'pageviews_90d',
            'ai_sessions_90d', 'scroll_events_90d', 'users_90d']
for col in log_cols:
    df[f'log_{col}'] = np.log1p(df[col].clip(lower=0).fillna(0))

# --- Ratio features ---
df['ctr_x_impressions'] = df['ctr'] * np.log1p(df['impressions_90d'].clip(lower=0))
df['engagement_x_sessions'] = df['engagement_rate'] * np.log1p(df['sessions_90d'].clip(lower=0))
df['scroll_x_ctr'] = df['scroll_rate'].fillna(0) * df['ctr'].fillna(0)
df['position_x_impressions'] = df['avg_position'].replace(0, np.nan) * np.log1p(df['impressions_90d'].clip(lower=0))

# --- Momentum features ---
prev_imp = df['impressions_prev_30d'].clip(lower=0)
last_imp = df['impressions_last_30d'].clip(lower=0)
df['impression_momentum'] = np.where(prev_imp > 0, last_imp / prev_imp, 0)
df['impression_momentum'] = df['impression_momentum'].clip(0, 10)  # clip extreme ratios

prev_clk = df['clicks_prev_30d'].clip(lower=0)
last_clk = df['clicks_last_30d'].clip(lower=0)
df['click_momentum'] = np.where(prev_clk > 0, last_clk / prev_clk, 0).clip(0, 10)

prev_sess = df['sessions_prev_30d'].clip(lower=0)
last_sess = df['sessions_last_30d'].clip(lower=0)
df['session_momentum'] = np.where(prev_sess > 0, last_sess / prev_sess, 0).clip(0, 10)

# --- Activity ratios ---
df['impressions_per_day'] = df['impressions_90d'] / df['days_with_impressions'].replace(0, np.nan)
df['sessions_per_day'] = df['sessions_90d'] / df['days_with_sessions'].replace(0, np.nan)
df['clicks_per_impression'] = df['clicks_90d'] / df['impressions_90d'].replace(0, np.nan)
df['engaged_session_rate'] = df['engaged_sessions_90d'] / df['sessions_90d'].replace(0, np.nan)
df['ai_session_share'] = df['ai_sessions_90d'] / df['sessions_90d'].replace(0, np.nan)
df['scroll_per_pageview'] = df['scroll_events_90d'] / df['pageviews_90d'].replace(0, np.nan)

# --- Content freshness ---
df['is_stale'] = (df['days_since_last_update'] > 90).astype(int)
df['is_very_stale'] = (df['days_since_last_update'] > 180).astype(int)
df['has_been_updated'] = (df['days_since_last_update'] < df['content_age_days']).astype(int)

# --- Position flags ---
df['has_position'] = (df['avg_position'] > 0).astype(int)
df['is_top10'] = ((df['avg_position'] > 0) & (df['avg_position'] <= 10)).astype(int)
df['is_top3'] = ((df['avg_position'] > 0) & (df['avg_position'] <= 3)).astype(int)

# --- Visibility flags ---
df['has_clicks'] = (df['clicks_90d'] > 0).astype(int)
df['has_ai_sessions'] = (df['ai_sessions_90d'] > 0).astype(int)
df['measurable_opportunity'] = ((df['impressions_90d'] >= 100) & (df['sessions_90d'] > 0)).astype(int)

# --- Fill NaN for engineered numeric features ---
engineered_numeric = [
    'ctr_x_impressions', 'engagement_x_sessions', 'scroll_x_ctr', 'position_x_impressions',
    'impression_momentum', 'click_momentum', 'session_momentum',
    'impressions_per_day', 'sessions_per_day', 'clicks_per_impression',
    'engaged_session_rate', 'ai_session_share', 'scroll_per_pageview',
    'position_x_impressions',
]
for col in engineered_numeric:
    df[col] = df[col].replace([np.inf, -np.inf], np.nan).fillna(0)

print(f'Engineered features: {len(engineered_numeric)}')
print(f'Total columns: {df.shape[1]}')

## 3. Define Final Feature Sets

In [ ]:
# Numeric features (excluding leakage)
NUMERIC_FEATURES = [
    # Raw traffic (log-transformed)
    'log_impressions_90d', 'log_clicks_90d', 'log_sessions_90d',
    'log_pageviews_90d', 'log_ai_sessions_90d', 'log_scroll_events_90d', 'log_users_90d',
    # Activity
    'days_with_impressions', 'days_with_sessions',
    # Rates
    'ctr', 'avg_position', 'engagement_rate', 'scroll_rate', 'ai_traffic_pct',
    # Content
    'word_count', 'char_count', 'content_age_days', 'days_since_last_update',
    'search_volume', 'competition', 'cpc',
    # Momentum
    'impression_momentum', 'click_momentum', 'session_momentum',
    # Ratios
    'ctr_x_impressions', 'engagement_x_sessions', 'scroll_x_ctr',
    'position_x_impressions', 'impressions_per_day', 'sessions_per_day',
    'clicks_per_impression', 'engaged_session_rate', 'ai_session_share', 'scroll_per_pageview',
    # Flags
    'is_stale', 'is_very_stale', 'has_been_updated',
    'has_position', 'is_top10', 'is_top3',
    'has_clicks', 'has_ai_sessions', 'measurable_opportunity',
]

# Categorical features
CATEGORICAL_FEATURES = [
    'competition_level', 'content_type', 'main_intent',
    'age_tier', 'freshness_tier', 'word_count_tier', 'char_count_tier',
    'impression_tier', 'position_tier',
]

# Verify no leakage
all_features = NUMERIC_FEATURES + CATEGORICAL_FEATURES
leakage_check = [c for c in all_features if c in LEAKAGE_COLS or 'trend' in c.lower() or 'decline' in c.lower()]
assert len(leakage_check) == 0, f'LEAKAGE DETECTED: {leakage_check}'
print(f'✓ Leakage check passed: {len(all_features)} features, no label-derived columns')
print(f'  Numeric: {len(NUMERIC_FEATURES)}')
print(f'  Categorical: {len(CATEGORICAL_FEATURES)}')

## 4. Prepare & Save Feature Vector

In [ ]:
# Fill categoricals
for col in CATEGORICAL_FEATURES:
    if col in df.columns:
        df[col] = df[col].fillna('unknown').astype(str).replace({'': 'unknown', 'nan': 'unknown'})
    else:
        df[col] = 'unknown'

# Fill remaining numeric NaN
for col in NUMERIC_FEATURES:
    if col in df.columns:
        df[col] = pd.to_numeric(df[col], errors='coerce').replace([np.inf, -np.inf], np.nan).fillna(0)
    else:
        df[col] = 0

# Save full feature vector
save_cols = ['content_id', 'client_id'] + all_features + ['is_declining', 'trend_class', 'trend_pct_filled',
           'decline_10pct', 'decline_20pct', 'decline_30pct',
           'impressions_90d', 'clicks_90d', 'sessions_90d', 'pageviews_90d',
           'engagement_rate', 'scroll_rate', 'avg_position', 'ctr',
           'days_since_last_update', 'content_age_days', 'content_type']
save_cols = [c for c in save_cols if c in df.columns]

feature_path = PROCESSED_DIR / 'capstone_feature_vector.csv'
df[save_cols].to_csv(feature_path, index=False)
print(f'Saved feature vector: {feature_path} ({len(save_cols)} columns, {len(df):,} rows)')

# Save feature config
config = {
    'NUMERIC_FEATURES': NUMERIC_FEATURES,
    'CATEGORICAL_FEATURES': CATEGORICAL_FEATURES,
    'LABELS': ['is_declining', 'trend_class', 'trend_pct_filled',
               'decline_10pct', 'decline_20pct', 'decline_30pct'],
    'LEAKAGE_COLS': LEAKAGE_COLS,
    'RANDOM_SEED': RANDOM_SEED,
}
config_path = PROCESSED_DIR / 'capstone_feature_config.json'
config_path.write_text(json.dumps(config, indent=2))
print(f'Saved feature config: {config_path}')

## 5. Quick Feature Importance Preview

In [ ]:
from sklearn.ensemble import RandomForestClassifier

# Encode categoricals for quick check
X_num = df[NUMERIC_FEATURES].fillna(0)
X_cat = pd.get_dummies(df[CATEGORICAL_FEATURES].fillna('unknown').astype(str), dummy_na=False, dtype=float)
X = pd.concat([X_num, X_cat], axis=1)
y = df['is_declining']

rf = RandomForestClassifier(n_estimators=100, max_depth=8, random_state=RANDOM_SEED, n_jobs=-1)
rf.fit(X, y)

imp = pd.DataFrame({'feature': X.columns, 'importance': rf.feature_importances_})
imp = imp.sort_values('importance', ascending=False).head(20)

fig, ax = plt.subplots(figsize=(10, 7))
ax.barh(imp['feature'][::-1], imp['importance'][::-1], color='#2A5D63')
ax.set_title('Top 20 Feature Importances (Quick RF Preview)')
ax.set_xlabel('Importance')
plt.tight_layout()
plt.savefig(ROOT / 'outputs' / 'charts' / 'feature_importance_preview.png', bbox_inches='tight')
plt.show()

print('Top 10:')
print(imp.head(10).to_string(index=False))